In [ ]:
# Copy this entire code into the first cell of VIT_7_Classes_Model.ipynb

# ===============================================================
# ENHANCED DERMNET CLASSIFICATION - VIT VERSION (TARGETED CLASSES)
# Loads ONLY the specific list of classes (previously excluded ones)
# ===============================================================

# ===============================================================
# BLOCK 1: Imports
# ===============================================================
import os, time, json, warnings
import numpy as np
import pandas as pd
from tqdm import tqdm
from PIL import Image
import cv2
import matplotlib.pyplot as plt
import seaborn as sns
warnings.filterwarnings("ignore")

import torch
import torch.nn as nn
import torch.optim as optim
import torch.nn.functional as F
from torch.utils.data import Dataset, DataLoader
from torchvision import transforms
from sklearn.preprocessing import LabelEncoder
from sklearn.model_selection import train_test_split
from sklearn.utils.class_weight import compute_class_weight
from sklearn.metrics import confusion_matrix, classification_report, f1_score
from transformers import ViTForImageClassification

device = torch.device("cuda" if torch.cuda.is_available() else "cpu")
print(f"Device: {device}")

# ===============================================================
# BLOCK 2: Enhanced CLAHE + Transforms
# ===============================================================
class ApplyCLAHE:
    def __init__(self, clipLimit=3.0, tileGridSize=(8,8)):
        self.clipLimit = clipLimit
        self.tileGridSize = tileGridSize
    def __call__(self, img: Image.Image):
        img_cv = cv2.cvtColor(np.array(img), cv2.COLOR_RGB2BGR)
        lab = cv2.cvtColor(img_cv, cv2.COLOR_BGR2LAB)
        l, a, b = cv2.split(lab)
        clahe = cv2.createCLAHE(clipLimit=self.clipLimit, tileGridSize=self.tileGridSize)
        cl = clahe.apply(l)
        limg = cv2.merge((cl, a, b))
        final = cv2.cvtColor(limg, cv2.COLOR_LAB2RGB)
        return Image.fromarray(final)

# Strong augmentation for all classes that need it
train_transform_strong = transforms.Compose([
    ApplyCLAHE(clipLimit=3.0),
    transforms.RandomResizedCrop(224, scale=(0.7, 1.0)),
    transforms.RandomHorizontalFlip(p=0.5),
    transforms.RandomVerticalFlip(p=0.3),
    transforms.RandomRotation(degrees=30),
    transforms.ColorJitter(brightness=0.3, contrast=0.3, saturation=0.2, hue=0.03),
    transforms.RandomAffine(degrees=0, translate=(0.1, 0.1)),
    transforms.ToTensor(),
    transforms.Normalize(mean=[0.485,0.456,0.406], std=[0.229,0.224,0.225]),
    transforms.RandomErasing(p=0.2)
])

train_transform = transforms.Compose([
    ApplyCLAHE(),
    transforms.RandomResizedCrop(224, scale=(0.8, 1.0)),
    transforms.RandomHorizontalFlip(),
    transforms.ColorJitter(brightness=0.2, contrast=0.2, saturation=0.15, hue=0.02),
    transforms.ToTensor(),
    transforms.Normalize(mean=[0.485,0.456,0.406], std=[0.229,0.224,0.225])
])

eval_transform = transforms.Compose([
    ApplyCLAHE(),
    transforms.Resize((224,224)),
    transforms.ToTensor(),
    transforms.Normalize(mean=[0.485,0.456,0.406], std=[0.229,0.224,0.225])
])

# ===============================================================
# BLOCK 3: Dataset Loading Functions (MODIFIED FOR TARGETING)
# ===============================================================
def load_images_and_labels(root_dir, target_classes=None):
    """
    Load images and labels from directory.
    Loads ONLY the classes present in 'target_classes'.
    """
    if target_classes is None:
        # If no list provided, assume we want everything (safety fallback)
        target_classes = sorted(os.listdir(root_dir))
        
    images, labels = [], []
    
    # List all folders
    all_classes = sorted(os.listdir(root_dir))
    
    print(f"Scanning {root_dir}...")
    
    for label in all_classes:
        # CHECK: Is this class in our TARGET list?
        if label not in target_classes:
            continue  # Skip this class because it's not in our list
            
        label_dir = os.path.join(root_dir, label)
        if os.path.isdir(label_dir):
            for fname in os.listdir(label_dir):
                fpath = os.path.join(label_dir, fname)
                if os.path.isfile(fpath) and fpath.lower().endswith(('.png', '.jpg', '.jpeg')):
                    images.append(fpath)
                    labels.append(label)
                    
    return images, labels

# ===============================================================
# BLOCK 4: Load and Combine Train/Test Data
# ===============================================================
print("\n" + "="*60)
print("LOADING DATA (TARGET CLASSES ONLY)")
print("="*60)

TRAIN_DIR = "/kaggle/input/dermnet/train"
TEST_DIR = "/kaggle/input/dermnet/test"

# Define the TARGET CLASSES (The ones previously excluded)
TARGET_CLASSES = [
    "Tinea Ringworm Candidiasis and other Fungal Infections",
    "Nail Fungus and other Nail Disease",
    "Actinic Keratosis Basal Cell Carcinoma and other Malignant Lesions",
    "Eczema Photos",
    "Warts Molluscum and other Viral Infections",
    "Seborrheic Keratoses and other Benign Tumors",
    "Psoriasis pictures Lichen Planus and related diseases",
]

print(f"Target Classes ({len(TARGET_CLASSES)}):")
for c in TARGET_CLASSES:
    print(f" - {c}")

# Load ONLY the targeted classes
print("\nLoading Train Data...")
train_images, train_labels = load_images_and_labels(TRAIN_DIR, target_classes=TARGET_CLASSES)

print("Loading Test Data...")
test_images, test_labels = load_images_and_labels(TEST_DIR, target_classes=TARGET_CLASSES)

print(f"Loaded train images: {len(train_images)}")
print(f"Loaded test images: {len(test_images)}")

# Combine all data
all_images = train_images + test_images
all_labels = train_labels + test_labels

# Encode labels
label_encoder = LabelEncoder()
all_encoded_labels = label_encoder.fit_transform(all_labels)
classes = list(label_encoder.classes_)
num_classes = len(classes)

print(f"Total images: {len(all_images)}")
print(f"Active Classes: {num_classes}")
print(f"Class Names: {classes}")

# Create combined dataframe
df_combined = pd.DataFrame({
    "path": all_images,
    "label": all_labels,
    "label_enc": all_encoded_labels
})

# ===============================================================
# BLOCK 5: Class Distribution (Before Augmentation)
# ===============================================================
print("\n" + "="*60)
print("CLASS DISTRIBUTION (BEFORE AUGMENTATION)")
print("="*60)

class_counts_before = df_combined['label'].value_counts().sort_index()
print(class_counts_before)

# Find the maximum class count to set as the target
# Since Psoriasis is in this list and has 4000, this target remains valid/correct.
max_class_count = 4000

print(f"\nTarget samples per class manually set to: {max_class_count}")

# ===============================================================
# BLOCK 6: Apply Augmentation to Balance Classes (MODIFIED)
# ===============================================================
print("\n" + "="*60)
print("APPLYING AUGMENTATION TO ALL CLASSES")
print("="*60)

augmented_data = []

for class_name in tqdm(classes, desc="Augmenting"):
    class_df = df_combined[df_combined['label'] == class_name]
    class_count = len(class_df)

    # Add original samples
    augmented_data.extend(class_df.to_dict('records'))

    # Calculate how many augmentations are needed to reach the max count
    num_augmentations_needed = max_class_count - class_count

    if num_augmentations_needed > 0:
        # To ensure variety, we sample more images than needed and then take what we need.
        # We sample with replacement if the class is very small.
        num_samples_to_pick = min(num_augmentations_needed * 2, len(class_df) * 5)

        samples_to_augment = list(class_df.sample(
            n=num_samples_to_pick,
            replace=True if num_samples_to_pick > len(class_df) else False,
            random_state=42 # for reproducibility
        ).to_dict('records'))

        # Add the augmented samples (up to the number needed)
        augmented_data.extend(samples_to_augment[:num_augmentations_needed])

        print(f"  - {class_name}: {class_count} -> {max_class_count} (+{num_augmentations_needed})")

df_augmented = pd.DataFrame(augmented_data)

# ===============================================================
# BLOCK 7: Class Distribution (After Augmentation) - MODIFIED
# ===============================================================
print("\n" + "="*60)
print("CLASS DISTRIBUTION (AFTER AUGMENTATION)")
print("="*60)

class_counts_after = df_augmented['label'].value_counts().sort_index()
print(class_counts_after)

comparison_df = pd.DataFrame({
    'Before': class_counts_before,
    'After': class_counts_after,
    'Increase': class_counts_after - class_counts_before,
    'Increase %': ((class_counts_after - class_counts_before) / class_counts_before * 100).round(1)
})
print("\n", comparison_df)

# Visualize the before and after
plt.figure(figsize=(15, 10))
plt.subplot(2, 1, 1)
class_counts_before.plot(kind='bar', color='skyblue')
plt.title('Class Distribution Before Augmentation')
plt.ylabel('Count')
plt.xticks(rotation=45, ha='right')

plt.subplot(2, 1, 2)
class_counts_after.plot(kind='bar', color='lightgreen')
plt.title(f'Class Distribution After Augmentation (All Classes at {max_class_count})')
plt.ylabel('Count')
plt.xticks(rotation=45, ha='right')

plt.tight_layout()
plt.savefig('class_distribution_before_after.png', dpi=300)
plt.show()

# ===============================================================
# BLOCK 8: Stratified Train/Val/Test Split
# ===============================================================
print("\n" + "="*60)
print("STRATIFIED SPLIT")
print("="*60)

# 80% train, 10% val, 10% test
train_df, temp_df = train_test_split(
    df_augmented, test_size=0.20, random_state=42, stratify=df_augmented['label_enc']
)

val_df, test_df = train_test_split(
    temp_df, test_size=0.50, random_state=42, stratify=temp_df['label_enc']
)

print(f"Train: {len(train_df)} ({len(train_df)/len(df_augmented)*100:.1f}%)")
print(f"Val: {len(val_df)} ({len(val_df)/len(df_augmented)*100:.1f}%)")
print(f"Test: {len(test_df)} ({len(test_df)/len(df_augmented)*100:.1f}%)")

# ===============================================================
# BLOCK 9: Custom Dataset Class
# ===============================================================
class DataFrameDataset(Dataset):
    def __init__(self, df, label_encoder, transform=None):
        self.df = df.reset_index(drop=True)
        self.label_encoder = label_encoder
        self.transform = transform

    def __len__(self):
        return len(self.df)

    def __getitem__(self, idx):
        row = self.df.iloc[idx]
        img = Image.open(row['path']).convert("RGB")
        label = int(row['label_enc'])

        if self.transform:
            img = self.transform(img)

        return img, label

train_dataset = DataFrameDataset(train_df, label_encoder, train_transform)
val_dataset = DataFrameDataset(val_df, label_encoder, eval_transform)
test_dataset = DataFrameDataset(test_df, label_encoder, eval_transform)

# ===============================================================
# BLOCK 10: Weighted Sampler for Class Balancing
# ===============================================================
print("\n" + "="*60)
print("COMPUTING CLASS WEIGHTS")
print("="*60)

y_train = train_df['label_enc'].values
class_weights_np = compute_class_weight(
    class_weight="balanced",
    classes=np.arange(num_classes),
    y=y_train
)

print(f"Class weights (first 5): {class_weights_np[:5]}")

from torch.utils.data import WeightedRandomSampler
sample_weights = class_weights_np[y_train]
sample_weights = torch.tensor(sample_weights, dtype=torch.double)
sampler = WeightedRandomSampler(
    weights=sample_weights,
    num_samples=len(sample_weights),
    replacement=True
)

# ===============================================================
# BLOCK 11: DataLoaders - MODIFIED
# ===============================================================
batch_size = 128

# Reduce workers for larger dataset to avoid I/O issues in Colab
num_workers = 2

train_loader = DataLoader(train_dataset, batch_size=batch_size, sampler=sampler,
                          num_workers=num_workers, pin_memory=True)
val_loader = DataLoader(val_dataset, batch_size=batch_size, shuffle=False,
                        num_workers=num_workers, pin_memory=True)
test_loader = DataLoader(test_dataset, batch_size=batch_size, shuffle=False,
                         num_workers=num_workers, pin_memory=True)

print(f"\nDataLoaders created:")
print(f"Train batches: {len(train_loader)}")
print(f"Val batches: {len(val_loader)}")
print(f"Test batches: {len(test_loader)}")

# ===============================================================
# BLOCK 12: Build ViT Model
# ===============================================================
print("\n" + "="*60)
print("LOADING VIT MODEL")
print("="*60)

model = ViTForImageClassification.from_pretrained(
    "google/vit-base-patch16-224",
    num_labels=num_classes,
    ignore_mismatched_sizes=True
)
model.to(device)

print(f"Model: ViT-Base")
print(f"Parameters: {sum(p.numel() for p in model.parameters()):,}")

# ===============================================================
# BLOCK 13: Loss, Optimizer, Scheduler
# ===============================================================
class_weights_tensor = torch.tensor(class_weights_np, dtype=torch.float).to(device)
criterion = nn.CrossEntropyLoss(weight=class_weights_tensor, label_smoothing=0.1)

optimizer = optim.AdamW(model.parameters(), lr=2e-4, weight_decay=1e-4)

scheduler = optim.lr_scheduler.CosineAnnealingWarmRestarts(
    optimizer, T_0=10, T_mult=2, eta_min=1e-6
)

# ===============================================================
# BLOCK 14: Training Functions
# ===============================================================
def calc_acc(logits, labels):
    preds = logits.argmax(dim=1)
    return (preds == labels).float().mean().item()

def train_epoch(model, loader, criterion, optimizer, device):
    model.train()
    total_loss, total_acc, batches = 0, 0, 0

    for images, labels in tqdm(loader, desc="Training"):
        images, labels = images.to(device), labels.to(device)

        optimizer.zero_grad()
        outputs = model(images).logits
        loss = criterion(outputs, labels)
        loss.backward()

        torch.nn.utils.clip_grad_norm_(model.parameters(), max_norm=1.0)
        optimizer.step()

        total_loss += loss.item()
        total_acc += calc_acc(outputs, labels)
        batches += 1

    return total_loss / batches, total_acc / batches

def validate(model, loader, criterion, device):
    model.eval()
    total_loss, total_acc, batches = 0, 0, 0
    all_preds, all_labels = [], []

    with torch.no_grad():
        for images, labels in tqdm(loader, desc="Validating"):
            images, labels = images.to(device), labels.to(device)
            outputs = model(images).logits
            loss = criterion(outputs, labels)

            total_loss += loss.item()
            total_acc += calc_acc(outputs, labels)
            batches += 1

            preds = torch.argmax(outputs, dim=1)
            all_preds.extend(preds.cpu().numpy())
            all_labels.extend(labels.cpu().numpy())

    avg_loss = total_loss / batches
    avg_acc = total_acc / batches
    f1 = f1_score(all_labels, all_preds, average='weighted')

    return avg_loss, avg_acc, f1, all_preds, all_labels

# ===============================================================
# BLOCK 15: Training Loop - MODIFIED
# ===============================================================
print("\n" + "="*60)
print("STARTING TRAINING")
print("="*60)

# Reduce epochs for the larger dataset to prevent overfitting
epochs = 12
patience = 3
best_val_f1 = 0.0
no_improve = 0

history = {
    "epoch": [], "train_loss": [], "val_loss": [],
    "train_acc": [], "val_acc": [], "val_f1": [], "lr": []
}

start_time = time.time()

for epoch in range(1, epochs + 1):
    print(f"\nEpoch {epoch}/{epochs}")
    print("-" * 60)

    # Train
    train_loss, train_acc = train_epoch(model, train_loader, criterion, optimizer, device)

    # Validate
    val_loss, val_acc, val_f1, val_preds, val_labels = validate(model, val_loader, criterion, device)

    # Scheduler
    current_lr = optimizer.param_groups[0]['lr']
    scheduler.step()

    # History
    history["epoch"].append(epoch)
    history["train_loss"].append(train_loss)
    history["val_loss"].append(val_loss)
    history["train_acc"].append(train_acc)
    history["val_acc"].append(val_acc)
    history["val_f1"].append(val_f1)
    history["lr"].append(current_lr)

    print(f"Train Loss: {train_loss:.4f} | Train Acc: {train_acc:.4f}")
    print(f"Val Loss: {val_loss:.4f} | Val Acc: {val_acc:.4f} | Val F1: {val_f1:.4f}")
    print(f"LR: {current_lr:.6f}")

    # Save best model
    if val_f1 > best_val_f1 + 1e-4:
        best_val_f1 = val_f1
        no_improve = 0

        checkpoint = {
            "epoch": epoch,
            "model_state_dict": model.state_dict(),
            "optimizer_state_dict": optimizer.state_dict(),
            "best_val_f1": best_val_f1,
            "history": history,
            "label_mapping": classes,
            "class_weights": class_weights_np.tolist()
        }
        torch.save(checkpoint, "best_vit_dermnet_balanced.pth")
        print(f"✓ Best model saved! (F1: {best_val_f1:.4f})")
    else:
        no_improve += 1
        if no_improve >= patience:
            print(f"\nEarly stopping at epoch {epoch}")
            break

total_time = time.time() - start_time
print(f"\nTraining completed in {total_time/60:.2f} minutes")
print(f"Best Val F1: {best_val_f1:.4f}")

# ===============================================================
# BLOCK 16: Plot Training Curves
# ===============================================================
fig, axes = plt.subplots(2, 2, figsize=(15, 10))

axes[0, 0].plot(history["epoch"], history["train_loss"], label="Train", linewidth=2)
axes[0, 0].plot(history["epoch"], history["val_loss"], label="Val", linewidth=2)
axes[0, 0].set_xlabel("Epoch"); axes[0, 0].set_ylabel("Loss")
axes[0, 0].set_title("Loss"); axes[0, 0].legend(); axes[0, 0].grid(True, alpha=0.3)

axes[0, 1].plot(history["epoch"], history["train_acc"], label="Train", linewidth=2)
axes[0, 1].plot(history["epoch"], history["val_acc"], label="Val", linewidth=2)
axes[0, 1].set_xlabel("Epoch"); axes[0, 1].set_ylabel("Accuracy")
axes[0, 1].set_title("Accuracy"); axes[0, 1].legend(); axes[0, 1].grid(True, alpha=0.3)

axes[1, 0].plot(history["epoch"], history["val_f1"], linewidth=2, color='green')
axes[1, 0].set_xlabel("Epoch"); axes[1, 0].set_ylabel("F1 Score")
axes[1, 0].set_title("Val F1 Score"); axes[1, 0].grid(True, alpha=0.3)

axes[1, 1].plot(history["epoch"], history["lr"], linewidth=2, color='red')
axes[1, 1].set_xlabel("Epoch"); axes[1, 1].set_ylabel("LR")
axes[1, 1].set_title("Learning Rate"); axes[1, 1].set_yscale('log')
axes[1, 1].grid(True, alpha=0.3)

plt.tight_layout()
plt.savefig("training_curves_vit_balanced.png", dpi=300)
plt.show()

# ===============================================================
# BLOCK 17: Load Best Model and Test
# ===============================================================
print("\n" + "="*60)
print("TESTING ON TEST SET")
print("="*60)

# Load with proper error handling
try:
    checkpoint = torch.load("best_vit_dermnet_balanced.pth", map_location=device, weights_only=False)
except TypeError:
    checkpoint = torch.load("best_vit_dermnet_balanced.pth", map_location=device)

model.load_state_dict(checkpoint["model_state_dict"])
label_names = checkpoint["label_mapping"]

# Test
test_loss, test_acc, test_f1, test_preds, test_labels = validate(model, test_loader, criterion, device)

print(f"\nTEST RESULTS:")
print(f"Test Loss: {test_loss:.4f}")
print(f"Test Accuracy: {test_acc:.4f} ({test_acc*100:.2f}%)")
print(f"Test F1 Score: {test_f1:.4f}")

# ===============================================================
# BLOCK 18: Confusion Matrix
# ===============================================================
print("\n" + "="*60)
print("CONFUSION MATRIX")
print("="*60)

cm = confusion_matrix(test_labels, test_preds)

plt.figure(figsize=(16, 14))
sns.heatmap(cm, annot=True, fmt='d', cmap='Blues',
            xticklabels=label_names, yticklabels=label_names)
plt.title('Confusion Matrix - Test Set', fontsize=16, fontweight='bold')
plt.xlabel('Predicted', fontsize=12)
plt.ylabel('True', fontsize=12)
plt.xticks(rotation=45, ha='right')
plt.yticks(rotation=0)
plt.tight_layout()
plt.savefig("confusion_matrix_vit_balanced.png", dpi=300)
plt.show()

# ===============================================================
# BLOCK 19: Classification Report
# ===============================================================
report = classification_report(test_labels, test_preds,
                               target_names=label_names,
                               output_dict=True,
                               zero_division=0)

report_df = pd.DataFrame(report).transpose()
print("\nClassification Report:")
print(report_df)

report_df.to_csv("classification_report_vit_balanced.csv")

# Per-class metrics
class_metrics = []
for i, class_name in enumerate(label_names):
    class_report = report[class_name]
    class_count = np.sum(np.array(test_labels) == i)

    class_metrics.append({
        'Class': class_name,
        'Samples': class_count,
        'Precision': class_report['precision'],
        'Recall': class_report['recall'],
        'F1-Score': class_report['f1-score']
    })

metrics_df = pd.DataFrame(class_metrics).sort_values('F1-Score', ascending=False)
print("\nPer-Class Performance:")
print(metrics_df.to_string(index=False))

# Plot per-class F1
plt.figure(figsize=(14, 8))
bars = plt.barh(metrics_df['Class'], metrics_df['F1-Score'])
plt.xlabel('F1-Score', fontsize=12)
plt.ylabel('Class', fontsize=12)
plt.title('Per-Class F1-Score', fontsize=14, fontweight='bold')
plt.xlim(0, 1.0)
plt.grid(axis='x', alpha=0.3)

for i, (idx, row) in enumerate(metrics_df.iterrows()):
    if row['F1-Score'] < 0.7:
        bars[i].set_color('indianred')
    elif row['F1-Score'] < 0.85:
        bars[i].set_color('orange')
    else:
        bars[i].set_color('forestgreen')

plt.tight_layout()
plt.savefig("per_class_f1_vit_balanced.png", dpi=300)
plt.show()

# ===============================================================
# BLOCK 20: Save Final Model
# ===============================================================
print("\n" + "="*60)
print("SAVING FINAL MODEL")
print("="*60)

final_save = {
    "model_state_dict": model.state_dict(),
    "label_mapping": label_names,
    "class_weights": class_weights_np.tolist(),
    "num_classes": num_classes,
    "test_accuracy": test_acc,
    "test_f1": test_f1
}

torch.save(final_save, "final_vit_model_balanced.pth")
print("✓ Saved: final_vit_model_balanced.pth")

with open("label_mapping_balanced.json", "w") as f:
    json.dump(label_names, f, indent=2)
print("✓ Saved: label_mapping_balanced.json")

history_df = pd.DataFrame(history)
history_df.to_csv("training_history_vit_balanced.csv", index=False)
print("✓ Saved: training_history_vit_balanced.csv")

summary = {
    "model": "ViT-Base-Patch16-224",
    "dataset": "DermNet (Fully Balanced)",
    "num_classes": num_classes,
    "max_class_count": int(max_class_count),
    "train_samples": len(train_df),
    "val_samples": len(val_df),
    "test_samples": len(test_df),
    "best_epoch": checkpoint['epoch'],
    "test_accuracy": float(test_acc),
    "test_f1": float(test_f1),
    "training_time_minutes": float(total_time / 60),
    "improvements": [
        "All classes augmented to maximum count",
        "Combined train/test for stratified split",
        "Advanced augmentation for all classes",
        "Class balancing with weighted sampler",
        "Label smoothing (0.1)",
        "Gradient clipping",
        "Cosine annealing scheduler",
        "Early stopping on F1 score"
    ]
}

with open("model_summary_vit_balanced.json", "w") as f:
    json.dump(summary, f, indent=2)
print("✓ Saved: model_summary_vit_balanced.json")

# ===============================================================
# BLOCK 21: Inference Function
# ===============================================================
def predict_image(image_path_or_pil, model, transform, label_names, device, top_k=3):
    """Predict class for a single image"""
    model.eval()

    if isinstance(image_path_or_pil, str):
        img = Image.open(image_path_or_pil).convert("RGB")
    else:
        img = image_path_or_pil

    img_tensor = transform(img).unsqueeze(0).to(device)

    with torch.no_grad():
        logits = model(img_tensor).logits
        probs = F.softmax(logits, dim=-1).cpu().numpy()[0]

    top_indices = np.argsort(probs)[::-1][:top_k]

    predictions = []
    for idx in top_indices:
        predictions.append({
            'class': label_names[idx],
            'confidence': float(probs[idx]),
            'class_id': int(idx)
        })

    return {
        'top_prediction': predictions[0],
        'all_predictions': predictions
    }

# Test inference
print("\n" + "="*60)
print("TESTING INFERENCE")
print("="*60)

sample_path = test_df.iloc[0]['path']
result = predict_image(sample_path, model, eval_transform, label_names, device)

print(f"✓ Inference successful!")
print(f"Top Prediction: {result['top_prediction']['class']}")
print(f"Confidence: {result['top_prediction']['confidence']:.4f}")
print(f"\nTop-3 Predictions:")
for i, pred in enumerate(result['all_predictions'], 1):
    print(f"  {i}. {pred['class']}: {pred['confidence']:.4f}")

# ===============================================================
# BLOCK 22: Gradio Interface (Optional)
# ===============================================================
print("\n" + "="*60)
print("GRADIO INTERFACE")
print("="*60)

try:
    import gradio as gr

    def gradio_predict(img_pil):
        result = predict_image(img_pil, model, eval_transform, label_names, device, top_k=5)

        label_dict = {pred['class']: pred['confidence'] for pred in result['all_predictions']}

        top_pred = result['top_prediction']
        info_text = f"**Class:** {top_pred['class']}\n**Confidence:** {top_pred['confidence']:.2%}"

        return label_dict, info_text

    demo = gr.Interface(
        fn=gradio_predict,
        inputs=gr.Image(type="pil", label="Upload Skin Image"),
        outputs=[
            gr.Label(num_top_classes=5, label="Top-5 Predictions"),
            gr.Markdown(label="Details")
        ],
        title="DermNet Skin Disease Classifier (Fully Balanced ViT)",
        description=f"""
        **Model:** ViT-Base Fine-tuned on Fully Balanced DermNet
        **Test Accuracy:** {test_acc:.2%}
        **Test F1-Score:** {test_f1:.4f}

        ⚠️ For educational purposes only.
        """,
        examples=[[test_df.iloc[i]['path']] for i in range(min(5, len(test_df)))],
        theme=gr.themes.Soft()
    )

    print("✓ Gradio interface ready!")
    print("  Run: demo.launch(share=True)")

    # Uncomment to launch:
    # demo.launch(share=False)

except ImportError:
    print("⚠️ Gradio not installed")
    print("  Install: pip install gradio")

# ===============================================================
# BLOCK 23: Final Summary
# ===============================================================
print("\n" + "="*60)
print("TRAINING COMPLETE!")
print("="*60)

print(f"""
FINAL RESULTS:
{'='*60}
Model:              ViT-Base-Patch16-224
Dataset:            DermNet (Fully Balanced)
Max Class Count:    {max_class_count}
Total Samples:      {len(df_augmented):,}
├─ Train:          {len(train_df):,} ({len(train_df)/len(df_augmented)*100:.1f}%)
├─ Validation:     {len(val_df):,} ({len(val_df)/len(df_augmented)*100:.1f}%)
└─ Test:           {len(test_df):,} ({len(test_df)/len(df_augmented)*100:.1f}%)

Best Epoch:         {checkpoint['epoch']}
Training Time:      {total_time/60:.2f} minutes

TEST PERFORMANCE:
├─ Accuracy:       {test_acc:.4f} ({test_acc*100:.2f}%)
├─ F1-Score:       {test_f1:.4f}
└─ Loss:           {test_loss:.4f}

IMPROVEMENTS APPLIED:
✓ All classes augmented to {max_class_count} samples
✓ Combined train/test for proper splitting
✓ Stratified train/val/test split
✓ Class balancing (WeightedRandomSampler)
✓ Label smoothing (0.1)
✓ Gradient clipping (max_norm=1.0)
✓ Cosine annealing scheduler
✓ Early stopping (F1-based)

FILES SAVED:
📁 best_vit_dermnet_balanced.pth
📁 final_vit_model_balanced.pth
📁 label_mapping_balanced.json
📁 training_history_vit_balanced.csv
📁 model_summary_vit_balanced.json
📁 classification_report_vit_balanced.csv
📁 confusion_matrix_vit_balanced.png
📁 per_class_f1_vit_balanced.png
📁 training_curves_vit_balanced.png
📁 class_distribution_before_after.png
{'='*60}

NEXT STEPS:
1. Compare with DinoV2 version for best results
2. Try ensemble of both models
3. Apply Test-Time Augmentation (TTA)
4. Use Grad-CAM for model explainability

Happy Training! 🚀
""")

print("="*60)


2025-12-11 22:45:12.547454: E external/local_xla/xla/stream_executor/cuda/cuda_fft.cc:477] Unable to register cuFFT factory: Attempting to register factory for plugin cuFFT when one has already been registered
E0000 00:00:1765493112.708754      47 cuda_dnn.cc:8310] Unable to register cuDNN factory: Attempting to register factory for plugin cuDNN when one has already been registered
E0000 00:00:1765493112.756104      47 cuda_blas.cc:1418] Unable to register cuBLAS factory: Attempting to register factory for plugin cuBLAS when one has already been registered
Device: cuda

LOADING DATA (TARGET CLASSES ONLY)
Target Classes (7):
 - Tinea Ringworm Candidiasis and other Fungal Infections
 - Nail Fungus and other Nail Disease
 - Actinic Keratosis Basal Cell Carcinoma and other Malignant Lesions
 - Eczema Photos
 - Warts Molluscum and other Viral Infections
 - Seborrheic Keratoses and other Benign Tumors
 - Psoriasis pictures Lichen Planus and related diseases

Loading Train Data...
Scanning /k